# 🧍 Neural Avatar Pipeline — Google Colab Training
> Run this notebook top to bottom to train your neural avatar model on a free GPU!

**Steps:**
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Find your mesh in `/content/neural-avatar-pipeline/checkpoints/`

## ✅ Step 1 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
!nvidia-smi

## 📦 Step 2 — Clone Your Repo

In [ ]:
!git clone https://github.com/S1o2u3/neural-avatar-pipeline.git
%cd neural-avatar-pipeline
!ls

## 🔧 Step 3 — Install Dependencies

In [ ]:
!pip install -r requirements.txt -q
!pip install scikit-image -q
print('✅ All dependencies installed!')

## 🧪 Step 4 — Test All Modules

In [ ]:
import sys
sys.path.insert(0, 'src')
import torch
from reconstruction.mesh_generator import MeshGenerator
from rendering.sdf_renderer import SDFRenderer
from utils.spatial_hash import SpatialHashGrid
from utils.ray_sampler import BatchedRaySampler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
mesh_gen     = MeshGenerator(device=device)
renderer     = SDFRenderer(device=device)
spatial_hash = SpatialHashGrid()
ray_sampler  = BatchedRaySampler(device=device)
mesh_gen.summary()
ray_sampler.summary()
print('✅ All modules loaded!')

## 🚀 Step 5 — Train the Model

In [ ]:
!python scripts/train.py \
    --dataset neuman \
    --epochs 50 \
    --batch_size 1024 \
    --steps_per_epoch 100 \
    --lr 5e-4 \
    --resolution 128 \
    --device cuda

## 💾 Step 6 — Download Your Mesh

In [ ]:
import sys, glob, torch, numpy as np
sys.path.insert(0, 'src')
from reconstruction.mesh_generator import MeshGenerator

device   = 'cuda' if torch.cuda.is_available() else 'cpu'
mesh_gen = MeshGenerator(device=device)
ckpts    = sorted(glob.glob('checkpoints/*.pt'))
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location=device)
    mesh_gen.load_state_dict(ckpt['model'])
    print(f'✅ Loaded: {ckpts[-1]}')

import os
os.makedirs('outputs', exist_ok=True)
verts, faces = mesh_gen.extract_mesh()
if verts is not None:
    np.save('outputs/vertices.npy', verts)
    np.save('outputs/faces.npy', faces)
    print(f'✅ Saved! Vertices: {len(verts)} Faces: {len(faces)}')

from google.colab import files
files.download('outputs/vertices.npy')
files.download('outputs/faces.npy')